# Explore one TikTok end-to-end

This deliberately small notebook downloads one public TikTok and persists both the MP4 and its public metadata. It tries **Pyktok first when browser cookies/DBus are available**, then reports its error before using the current yt-dlp release as a narrow fallback. On headless Linux/Jupyter, Pyktok is skipped explicitly because browser-cookie3 cannot read cookies without DBus. TikTok changes frequently, so failures include their original exception and are never ignored.

Setup (Python 3.10+ recommended): run the next cell once. Pyktok may need cookies from a locally installed Chrome/Firefox profile; set `BROWSER` below if necessary. Only download content you are entitled to collect and follow TikTok's terms and applicable law.

In [ ]:
%pip install -q --upgrade "pyktok==0.0.31" "yt-dlp>=2026.07.04"


In [ ]:
import json
import os
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import Video, display

# Public example from Pyktok's own documentation. Replace this one value to explore another video.
TIKTOK_URL = "https://www.tiktok.com/@tiktok/video/7106594312292453675"
BROWSER = os.environ.get("PYKTOK_BROWSER", "chrome")  # e.g. chrome or firefox
# In headless Linux/Jupyter environments browser-cookie3 can require DBus just to read cookies.
# Set PYKTOK_FORCE=1 only when you have configured browser cookies and want to force Pyktok.
PYKTOK_FORCE = os.environ.get("PYKTOK_FORCE", "0") == "1"

match = re.search(r"/video/(\d+)", TIKTOK_URL)
if not match:
    raise ValueError(f"Expected a canonical TikTok URL containing /video/<id>: {TIKTOK_URL}")
video_id = match.group(1)
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
output_dir = repo_root / "data" / "exploration" / video_id
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / "video.mp4"
metadata_path = output_dir / "metadata.json"
print(f"Source: {TIKTOK_URL}\nOutput: {output_dir.resolve()}")


## Download and collect metadata

Pyktok writes the MP4 beside its CSV, so the cell runs it inside the final exploration directory and then gives the file the stable name `video.mp4`. If Pyktok fails, the full error is printed before `yt-dlp` is attempted. If both fail, the cell raises a combined, actionable error.

In [ ]:
def json_safe(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value)

def first_present(data, *keys):
    lowered = {str(k).lower(): v for k, v in data.items()}
    for key in keys:
        value = lowered.get(key.lower())
        if value is not None and str(value) not in {"", "nan", "None"}:
            return json_safe(value)
    return None

def publication_value(raw):
    value = first_present(raw, "video_timestamp", "create_time", "timestamp", "upload_date")
    if isinstance(value, (int, float)) and value > 10**9:
        return datetime.fromtimestamp(value, tz=timezone.utc).isoformat()
    if isinstance(value, str) and value.isdigit() and int(value) > 10**9:
        return datetime.fromtimestamp(int(value), tz=timezone.utc).isoformat()
    if isinstance(value, str) and len(value) == 8 and value.isdigit():
        return f"{value[:4]}-{value[4:6]}-{value[6:]}"
    return value

def normalize_metadata(raw, collector):
    caption = first_present(raw, "video_description", "description", "desc", "title")
    hashtags = first_present(raw, "video_hashtags", "hashtags", "tags")
    if isinstance(hashtags, str):
        hashtags = re.findall(r"#([\w.-]+)", hashtags) or [x.strip() for x in hashtags.split(",") if x.strip()]
    if not hashtags and caption:
        hashtags = re.findall(r"#([\w.-]+)", caption)
    return {
        "source_url": TIKTOK_URL,
        "video_id": video_id,
        "collector": collector,
        "collected_at": datetime.now(timezone.utc).isoformat(),
        "creator": first_present(raw, "author_name", "author", "uploader", "creator", "author_uniqueid"),
        "caption": caption,
        "hashtags": hashtags or [],
        "views": first_present(raw, "video_playcount", "playcount", "view_count"),
        "likes": first_present(raw, "video_diggcount", "diggcount", "like_count"),
        "comments": first_present(raw, "video_commentcount", "commentcount", "comment_count"),
        "shares": first_present(raw, "video_sharecount", "sharecount", "repost_count"),
        "duration_seconds": first_present(raw, "video_duration", "duration"),
        "publication_date": publication_value(raw),
        "raw": {str(k): json_safe(v) for k, v in raw.items()},
    }

def collect_with_pyktok():
    import pandas as pd
    import pyktok as pyk
    csv_path = output_dir / "pyktok_metadata.csv"
    before = set(output_dir.glob("*.mp4"))
    previous_cwd = Path.cwd()
    try:
        os.chdir(output_dir)
        pyk.specify_browser(BROWSER)
        pyk.save_tiktok(TIKTOK_URL, True, csv_path.name, BROWSER)
    finally:
        os.chdir(previous_cwd)
    candidates = [p for p in output_dir.glob("*.mp4") if p not in before]
    if not candidates and video_path.exists():
        candidates = [video_path]
    if not candidates:
        raise FileNotFoundError(f"Pyktok returned without creating an MP4 in {output_dir}")
    downloaded = max(candidates, key=lambda p: p.stat().st_mtime)
    if downloaded != video_path:
        downloaded.replace(video_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"Pyktok returned without creating metadata CSV: {csv_path}")
    rows = pd.read_csv(csv_path).to_dict(orient="records")
    if not rows:
        raise ValueError(f"Pyktok metadata CSV is empty: {csv_path}")
    return normalize_metadata(rows[-1], "pyktok")

def collect_with_ytdlp():
    from yt_dlp import YoutubeDL
    options = {
        "outtmpl": str(video_path),
        "format": "best[ext=mp4]/best",
        "noplaylist": True,
        "quiet": False,
    }
    with YoutubeDL(options) as ydl:
        info = ydl.extract_info(TIKTOK_URL, download=True)
    return normalize_metadata(info, "yt-dlp (explicit fallback after Pyktok failure)")

pyktok_error = None
metadata = None
if not PYKTOK_FORCE and not os.environ.get("DBUS_SESSION_BUS_ADDRESS"):
    pyktok_error = (
        "Pyktok skipped: DBUS_SESSION_BUS_ADDRESS is not set, so browser-cookie3 cannot read cookies in this headless environment. Set PYKTOK_FORCE=1 after configuring browser cookies to retry it."
    )
    print(pyktok_error)
else:
    try:
        metadata = collect_with_pyktok()
    except Exception as exc:
        pyktok_error = f"{type(exc).__name__}: {exc}"
        print(f"Pyktok failed explicitly: {pyktok_error}")

if metadata is None:
    print("Trying the documented yt-dlp fallback...")
    try:
        metadata = collect_with_ytdlp()
        if pyktok_error:
            metadata["pyktok_error"] = pyktok_error
    except Exception as fallback_exc:
        raise RuntimeError(
            "Both collectors failed. If DBus is unavailable, set PYKTOK_FORCE=1 with configured browser cookies; otherwise update yt-dlp. "
            f"Pyktok: {pyktok_error or 'not attempted'}; yt-dlp: {type(fallback_exc).__name__}: {fallback_exc}"
        ) from fallback_exc

if not video_path.exists() or video_path.stat().st_size == 0:
    raise RuntimeError(f"Collector completed but MP4 is missing or empty: {video_path}")
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {video_path} ({video_path.stat().st_size / 1_048_576:.1f} MiB)")
print(f"Saved {metadata_path}")


## Manual verification

Compare the source URL/video ID, creator and caption below with the embedded MP4. The complete collector response remains under `raw` in `metadata.json` for debugging.

In [ ]:
summary_fields = ["source_url", "video_id", "collector", "creator", "caption", "hashtags", "views", "likes", "comments", "shares", "duration_seconds", "publication_date"]
print(json.dumps({key: metadata.get(key) for key in summary_fields}, ensure_ascii=False, indent=2))
display(Video(filename=str(video_path), embed=True, html_attributes="controls"))


In [ ]:
# Final persisted-artifact checks (fail loudly if the run is incomplete).
persisted = json.loads(metadata_path.read_text(encoding="utf-8"))
assert persisted["source_url"] == TIKTOK_URL
assert persisted["video_id"] == video_id
assert video_path.stat().st_size > 0
print("Verification passed:", video_path.resolve(), metadata_path.resolve())
